# Notebook 05 — Inferencia: Usar el Modelo Entrenado

> **Blog Técnico** · Pix2Pix: Demo interactivo de traducción satelite ↔ mapa

Este notebook carga los pesos entrenados desde `checkpoints/` y genera traducciones sobre imágenes nuevas.

**No se necesita re-entrenar.** Solo se carga el generador (`G`), cuyo estado está guardado en el `.pth`. El discriminador solo se usa durante el entrenamiento.

---
### ¿Qué contiene un checkpoint?

```python
{
  'epoca':         int,            # última época entrenada
  'generador':     state_dict,     # pesos de U-Net (lo que necesitamos aquí)
  'discriminador': state_dict,     # pesos de PatchGAN (no necesario en inferencia)
  'optimizador_g': state_dict,     # momentos de Adam G
  'optimizador_d': state_dict,     # momentos de Adam D
  'grad_scaler':   state_dict,     # estado del AMP scaler
  'losses_historia': dict,         # historial de pérdidas por época
}
```

In [ ]:
import sys, os, glob
from pathlib import Path

PROYECTO = Path('..').resolve() if Path('../src').exists() else Path('.')
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from src.models.generator import GeneradorUNet
from src.data.dataset_loader import DatasetParesSideBySide, desnormalizar
from src.data.transforms import TransformPar

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

## 1. Encontrar y Cargar el Checkpoint

In [ ]:
# ── Detectar automáticamente el checkpoint más reciente ───────────────────────
DIRECCION = 'AtoB'  # cambiar a 'BtoA' para el modelo satelite-generador

# Buscar en el directorio de checkpoints
patron = f'checkpoints/**/*{DIRECCION}*.pth'
checkpoints = sorted(glob.glob(patron, recursive=True))

if not checkpoints:
    raise FileNotFoundError(
        f"No se encontraron checkpoints con patrón '{patron}'.\n"
        f"Verifica que el entrenamiento completó al menos 1 época.\n"
        f"Checkpoints en: {list(Path('checkpoints').glob('**/*.pth'))}"
    )

RUTA_CKPT = checkpoints[-1]  # último = mayor número de época
print(f"Checkpoints disponibles: {len(checkpoints)}")
for p in checkpoints[-5:]:
    print(f"  {p}")
print(f"\nCargando: {RUTA_CKPT}")

In [ ]:
# ── Cargar el Generador ────────────────────────────────────────────────────────
G = GeneradorUNet(
    canales_entrada=3,
    canales_salida=3,
    filtros_base=64,
).to(device)

estado = torch.load(RUTA_CKPT, map_location=device)
G.load_state_dict(estado['generador'])
G.eval()  # Desactivar Dropout y BatchNorm en modo eval

epoca_cargada = estado.get('epoca', '?')
print(f"Generador cargado — época {epoca_cargada}")
print(f"Parámetros: {sum(p.numel() for p in G.parameters()):,}")

# Verificar que funciona con un tensor dummy
with torch.no_grad():
    x_test = torch.randn(1, 3, 256, 256).to(device)
    y_test = G(x_test)
assert y_test.shape == (1, 3, 256, 256)
assert y_test.min() >= -1.0 - 1e-4 and y_test.max() <= 1.0 + 1e-4
print(f"Forward pass OK: {tuple(y_test.shape)}, rango [{y_test.min():.3f}, {y_test.max():.3f}]")

## 2. Inferencia sobre el Conjunto de Validación

Evaluamos sobre imágenes del split `val/` — imágenes que el modelo **nunca vio** durante el entrenamiento.

In [ ]:
# ── Cargar dataset de validación ──────────────────────────────────────────────
dataset_val = DatasetParesSideBySide(
    directorio_raiz='data/processed/val',
    direction=DIRECCION,
    modo='val',
    cache_en_ram=False,
)
print(f"Dataset val: {len(dataset_val)} pares")

# ── Generar predicciones para 6 muestras espaciadas ───────────────────────────
IDXS = [0, 50, 100, 200, 400, 700]  # índices espaciados del val set

resultados = []
for idx in IDXS:
    if idx >= len(dataset_val):
        continue
    tensor_a, tensor_b = dataset_val[idx]
    tensor_a_batch = tensor_a.unsqueeze(0).to(device)  # (1,3,256,256)

    with torch.no_grad():
        tensor_fake_b = G(tensor_a_batch)

    resultados.append((
        desnormalizar(tensor_a).permute(1,2,0).numpy(),          # entrada
        desnormalizar(tensor_fake_b.squeeze(0).cpu()).permute(1,2,0).numpy(),  # generada
        desnormalizar(tensor_b).permute(1,2,0).numpy(),          # real objetivo
    ))

print(f"{len(resultados)} imágenes generadas.")

In [ ]:
# ── Visualizar: Entrada | Generado | Real ─────────────────────────────────────
n = len(resultados)
fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))
fig.suptitle(
    f'Resultados del modelo (época {epoca_cargada}) — {DIRECCION}\n'
    f'Columnas: Entrada  |  Generado por G  |  Real (objetivo)',
    fontsize=12, y=1.01
)

cols = ['Entrada (dominio A)', 'Generado por G', 'Real (dominio B)']

for i, (img_a, img_fake, img_b) in enumerate(resultados):
    for j, (img, col) in enumerate([(img_a, cols[0]), (img_fake, cols[1]), (img_b, cols[2])]):
        ax = axes[i, j] if n > 1 else axes[j]
        ax.imshow(np.clip(img, 0, 1))
        if i == 0:
            ax.set_title(col, fontsize=10, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
ruta_grid = f'results/inferencia_{DIRECCION}_ep{epoca_cargada}.png'
plt.savefig(ruta_grid, dpi=130, bbox_inches='tight')
plt.show()
print(f"Grid guardado en: {ruta_grid}")

## 3. Inferencia sobre una Imagen Propia

Puedes subir cualquier imagen satelital (o mapa, si el modelo es BtoA) y obtener su traducción.

In [ ]:
import torchvision.transforms.functional as TF

def inferir_imagen(ruta_imagen: str, generador: torch.nn.Module,
                  dispositivo: torch.device) -> np.ndarray:
    """
    Carga una imagen, la preprocesa y genera su traducción.

    Args:
        ruta_imagen: Ruta a la imagen de entrada (.jpg, .png, etc.)
        generador:   Modelo G en modo eval.
        dispositivo: torch.device

    Returns:
        Array numpy HxWx3 con la imagen generada en [0, 1].
    """
    img = PILImage.open(ruta_imagen).convert('RGB')

    # Pipeline de preprocesamiento idéntico al DataLoader val
    img_resized = TF.resize(img, [256, 256])
    tensor = TF.to_tensor(img_resized)              # [0, 1]
    tensor = TF.normalize(tensor, [0.5,0.5,0.5],   # → [-1, 1]
                                  [0.5,0.5,0.5])
    tensor = tensor.unsqueeze(0).to(dispositivo)    # (1,3,256,256)

    with torch.no_grad():
        salida = generador(tensor)

    # Desnormalizar: [-1,1] → [0,1]
    salida_np = ((salida.squeeze(0).cpu() + 1.0) / 2.0).clamp(0, 1)
    return salida_np.permute(1, 2, 0).numpy()


# ── Ejemplo: usar una imagen del val set como imagen propia ───────────────────
# Para usar tu imagen: cambia RUTA_IMAGEN a la ruta de tu archivo
ruta_ejemplo = sorted(Path('data/processed/val').glob('*.jpg'))[0]
img_original = PILImage.open(ruta_ejemplo)
w_medio = img_original.width // 2
img_satelital = img_original.crop((0, 0, w_medio, img_original.height))
ruta_tmp = '/tmp/mi_imagen_satelital.jpg'
img_satelital.save(ruta_tmp)

# Inferencia
img_generada = inferir_imagen(ruta_tmp, G, device)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_satelital.resize((256, 256)))
axes[0].set_title('Imagen de entrada (satelital)', fontsize=10)
axes[0].axis('off')

axes[1].imshow(np.clip(img_generada, 0, 1))
axes[1].set_title(f'Mapa generado por G (época {epoca_cargada})', fontsize=10)
axes[1].axis('off')

plt.suptitle('Inferencia sobre imagen individual', fontsize=12)
plt.tight_layout()
plt.savefig('results/inferencia_individual.png', dpi=130)
plt.show()

## 4. Evolución a lo Largo del Entrenamiento

Visualizamos cómo mejora la calidad de las imágenes generadas a lo largo de las épocas.

In [ ]:
# Cargar todas las muestras guardadas durante el entrenamiento
dir_resultados = Path('results') / Path(RUTA_CKPT).parent.name
muestras_guardadas = sorted(dir_resultados.glob('muestra_epoca_*.png'))

if not muestras_guardadas:
    # Buscar en todos los directorios de resultados
    muestras_guardadas = sorted(Path('results').glob('**/muestra_epoca_*.png'))

print(f"Muestras guardadas durante entrenamiento: {len(muestras_guardadas)}")

if len(muestras_guardadas) >= 2:
    # Mostrar primera, media y última
    idx_mostrar = [0, len(muestras_guardadas)//2, -1]
    fig, axes = plt.subplots(1, len(idx_mostrar), figsize=(16, 4))
    fig.suptitle('Evolución del generador durante el entrenamiento', fontsize=12)

    for ax, idx in zip(axes, idx_mostrar):
        ruta = muestras_guardadas[idx]
        with PILImage.open(ruta) as img:
            ax.imshow(img)
        # Extraer número de época del nombre del archivo
        ep = ruta.stem.split('_')[-1]
        ax.set_title(f'Época {ep}', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('results/evolucion_entrenamiento.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("Se necesitan al menos 2 muestras para mostrar la evolución.")